# H1 — Behavioral Effects of Threat on Choice, Effort, and Affect

**Prediction:** Threat will reduce high-effort choice, increase excess motor effort, and shift subjective anxiety upward and confidence downward.

**Sub-hypotheses:**
- **H1a:** High-effort choice will decrease with threat probability and with escape distance (logistic mixed model)
- **H1b:** Excess effort (pressing rate minus demand), conditioned on cookie type, will increase with threat probability (LMM)
- **H1c:** Trial-level anxiety will increase and confidence will decrease with threat probability T (LMMs)

**What this determines:** Whether threat has the expected directional effects on the three behavioral channels (choice, vigor, affect) before any computational modeling.

In [ ]:
# ── Imports & Data Loading (self-contained) ──
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, zscore
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.right': False, 'axes.spines.top': False,
})

DATA_DIR = Path("/workspace/data/exploratory_350/processed/stage5_filtered_data_20260320_191950")
OUT_DIR  = Path("/workspace/results/stats/full_analysis")
EXCLUDE  = [154, 197, 208]

# ── Choice trials (free-choice behavioral trials only) ──
beh = pd.read_csv(DATA_DIR / "behavior.csv")
beh = beh[~beh['subj'].isin(EXCLUDE)].copy()
beh['T_round'] = beh['threat'].round(1)
beh['threat_z'] = zscore(beh['threat'])
beh['dist_z']   = zscore(beh['distance_H'])
beh['trial_number'] = beh.groupby('subj').cumcount() + 1

# ── All trials (81 per subject, includes probes) ──
beh_rich = pd.read_csv(DATA_DIR / "behavior_rich.csv", low_memory=False)
beh_rich = beh_rich[~beh_rich['subj'].isin(EXCLUDE)].copy()
beh_rich['T_round'] = beh_rich['threat'].round(1)
beh_rich['is_heavy'] = (beh_rich['trialCookie_weight'] == 3.0).astype(int)
beh_rich['actual_req'] = np.where(beh_rich['is_heavy'] == 1, 0.9, 0.4)
beh_rich['enc_t'] = pd.to_numeric(beh_rich['encounterTime'], errors='coerce')

# ── Feelings (probe trials) ──
feelings = pd.read_csv(DATA_DIR / "feelings.csv")
feelings = feelings[~feelings['subj'].isin(EXCLUDE)].copy()

print(f"Choice trials: {len(beh):,} ({beh['subj'].nunique()} subjects)")
print(f"All trials:    {len(beh_rich):,} ({beh_rich['subj'].nunique()} subjects)")
print(f"Probe ratings: {len(feelings):,}")

## H1a — Choice decreases with threat and distance

**Model:** `choice ~ threat_z + dist_z + threat_z:dist_z + (1 | subj)`  
**Test:** beta(threat) < 0 AND beta(dist) < 0, both p < 0.01

In [ ]:
# ── H1a: Logistic mixed model — choice ~ threat + distance ──
model_h1a = smf.logit(
    "choice ~ threat_z + dist_z + threat_z:dist_z",
    data=beh
).fit(disp=False, cov_type='cluster', cov_kwds={'groups': beh['subj']})

print("H1a — Logistic regression (cluster-robust SE by subject)")
print("=" * 65)
print(model_h1a.summary2().tables[1][['Coef.', 'Std.Err.', 'z', 'P>|z|']].to_string())
print()

# Verdict
p_threat = model_h1a.pvalues['threat_z']
p_dist   = model_h1a.pvalues['dist_z']
b_threat = model_h1a.params['threat_z']
b_dist   = model_h1a.params['dist_z']
print(f"Threat:   beta = {b_threat:.4f}, p = {p_threat:.2e}  {'PASS' if b_threat < 0 and p_threat < 0.01 else 'FAIL'}")
print(f"Distance: beta = {b_dist:.4f}, p = {p_dist:.2e}  {'PASS' if b_dist < 0 and p_dist < 0.01 else 'FAIL'}")

In [ ]:
# ── H1a figure: 3x3 Choice Surface Heatmap ──
choice_surface = beh.groupby(['T_round', 'distance_H'])['choice'].mean().unstack()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(choice_surface.values, cmap='RdYlGn', vmin=0, vmax=1,
               origin='lower', aspect='auto')
ax.set_xticks(range(3))
ax.set_xticklabels(['D=1', 'D=2', 'D=3'])
ax.set_yticks(range(3))
ax.set_yticklabels(['T=0.1', 'T=0.5', 'T=0.9'])
ax.set_xlabel('Escape Distance')
ax.set_ylabel('Threat Probability')
ax.set_title('P(Choose High-Effort Cookie)')

# Add text annotations
for i in range(3):
    for j in range(3):
        val = choice_surface.values[i, j]
        color = 'white' if val < 0.4 or val > 0.8 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='P(Heavy)')
plt.tight_layout()
plt.savefig(OUT_DIR / "H1a_choice_surface.png", dpi=150, bbox_inches='tight')
plt.show()
print("Choice surface saved.")

## H1b — Excess effort increases with threat

**Model:** `excess_cc ~ threat_z + (1 | subj)` where excess effort = median(1/IPI)/calibrationMax - demand, centered by cookie type  
**Test:** beta(threat) > 0, p < 0.05

In [ ]:
# ── H1b: Excess effort ~ threat ──
# Compute normalized vigor per trial: median(1/IPI) / calibrationMax
def compute_trial_vigor(row):
    """Compute median(1/IPI)/calibrationMax for a trial."""
    try:
        pts = np.array(ast.literal_eval(str(row['alignedEffortRate'])), dtype=float)
        cal = row['calibrationMax']
        if len(pts) < 5 or cal <= 0:
            return np.nan
        ipis = np.diff(pts)
        ipis = ipis[ipis > 0.01]
        if len(ipis) < 3:
            return np.nan
        return np.median(1.0 / ipis) / cal
    except:
        return np.nan

beh_rich['norm_vigor'] = beh_rich.apply(compute_trial_vigor, axis=1)
beh_rich['excess_effort'] = beh_rich['norm_vigor'] - beh_rich['actual_req']

# Cookie-type centering: subtract mean excess per cookie type
cookie_means = beh_rich.groupby('is_heavy')['excess_effort'].transform('mean')
beh_rich['excess_cc'] = beh_rich['excess_effort'] - cookie_means
beh_rich['threat_z'] = zscore(beh_rich['threat'].astype(float))

# Fit LMM
excess_data = beh_rich.dropna(subset=['excess_cc']).copy()
model_h1b = smf.mixedlm(
    "excess_cc ~ threat_z",
    data=excess_data,
    groups=excess_data['subj']
).fit(reml=False)

print("H1b — LMM: excess effort (cookie-centered) ~ threat")
print("=" * 55)
print(f"  threat_z: beta = {model_h1b.fe_params['threat_z']:.4f}, "
      f"SE = {model_h1b.bse_fe['threat_z']:.4f}, "
      f"z = {model_h1b.tvalues['threat_z']:.3f}, "
      f"p = {model_h1b.pvalues['threat_z']:.4f}")
print()

b = model_h1b.fe_params['threat_z']
p = model_h1b.pvalues['threat_z']
print(f"Verdict: beta = {b:.4f}, p = {p:.4f}  {'PASS' if b > 0 and p < 0.05 else 'FAIL'}")

## H1c — Anxiety increases and confidence decreases with threat

**Model:** `anxiety ~ T_z + (1 + T_z | subj)` and `confidence ~ T_z + (1 + T_z | subj)`  
**Test:** beta(T) > 0 for anxiety and beta(T) < 0 for confidence, both |t| > 3.0

In [ ]:
# ── H1c: Affect ~ threat ──
# Split by question type
anx = feelings[feelings['questionType'] == 'anxiety'].copy()
con = feelings[feelings['questionType'] == 'confidence'].copy()
anx['T_z'] = zscore(anx['threat'].astype(float))
con['T_z'] = zscore(con['threat'].astype(float))

print("H1c — Affect models")
print("=" * 55)

for label, df in [('Anxiety', anx), ('Confidence', con)]:
    try:
        m = smf.mixedlm(
            "response ~ T_z", data=df, groups=df['subj'],
            re_formula="~T_z"
        ).fit(reml=False)
    except:
        m = smf.mixedlm(
            "response ~ T_z", data=df, groups=df['subj']
        ).fit(reml=False)

    b = m.fe_params['T_z']
    z = m.tvalues['T_z']
    p = m.pvalues['T_z']
    print(f"  {label:12s}: beta = {b:.4f}, z = {z:.3f}, p = {p:.2e}")
    if label == 'Anxiety':
        passed = b > 0 and abs(z) > 3.0
    else:
        passed = b < 0 and abs(z) > 3.0
    print(f"               {'PASS' if passed else 'FAIL'} (expect {'positive' if label == 'Anxiety' else 'negative'} beta, |t| > 3)")
    print()

## Summary

| Test | Prediction | Statistic | p-value | Verdict |
|------|-----------|-----------|---------|---------|
| H1a (threat→choice) | beta < 0 | _fill from output_ | _fill_ | _PASS/FAIL_ |
| H1a (distance→choice) | beta < 0 | _fill from output_ | _fill_ | _PASS/FAIL_ |
| H1b (threat→excess effort) | beta > 0 | _fill from output_ | _fill_ | _PASS/FAIL_ |
| H1c (threat→anxiety) | beta > 0, \|t\| > 3 | _fill from output_ | _fill_ | _PASS/FAIL_ |
| H1c (threat→confidence) | beta < 0, \|t\| > 3 | _fill from output_ | _fill_ | _PASS/FAIL_ |

**Interpretation:** If all pass, the basic behavioral effects are confirmed — threat suppresses risky choice, boosts motor output, and modulates subjective affect as expected. This establishes the foundation for computational modeling (H2+).